# 1. pre-process

Input: 
1. <dataset_name>_participants.tsv: 
dataset	| participant_id | participant_exp (in original dataset)| age_months (postnatal)	
BCP     | sub-001        | NCBCP388560_v01-1-2wk-20170309       | 0.46
2. T1w and T2w images:
for each row in <dataset_name>_participants.tsv: 
row['participant_id']_T1w.nii.gz, row['participant_id']_T2w.nii.gz


Note:
For now, only includes subjects with gestational age >=37 weeks


Process:
1. iBEAT (16h/subject for BCP, 1.5~3h for dHCP)
2. direct transfer (QC check, basic for indirect transfer)


## 1.1 iBEAT 2.0 to skull stripping
Visual QC

## 1.2 pad or reorient if necessary

In [8]:
import os
import sys
from tqdm import tqdm
import pandas as pd
Info_dir = '/project/4290000.01/yapwan/Projects/NeonatalNormalization/Info'
pipel_dir = "/project/4290000.01/yapwan/Projects/NeonatalNormalization/Data"
tpl_root = "/project/4290000.01/yapwan/toolbox/BCP-atlas-for_release-Ver2.0.0"
tpl_dirs = [d for d in os.listdir(tpl_root) if os.path.isdir(os.path.join(tpl_root, d))]
tpl_months = sorted([d.replace("Month", "") for d in tpl_dirs if d.endswith("Month")], key=int)

base_dir = '/project/4290000.01/yapwan/Projects/NeonatalNormalization/Script'
sys.path.append(base_dir)
from tpl_xfm_build import com_initialize_to_template_space

age_bins = ['12-15M', '15-18M', '18-21M', '21-24M'] # '24-36M', '36-48M', '48-60M', '60-216M'
info_age = pd.read_csv(os.path.join(Info_dir, f'BCP_HCPD_participants_0-18Y.tsv'), sep='\t')
for age_bin in age_bins:
    print(f"Processing age bin: {age_bin}")
    subs = (info_age.loc[info_age[age_bin], ["participant_id"]].drop_duplicates().values.flatten())
    print(f"Subjects in age bin {age_bin}: {len(subs)}")
    # for subid in tqdm(subs_pos):
    for subid in tqdm(subs):
        info_cur = info_age[info_age['participant_id']==subid].iloc[0]
        print(f"{info_cur['dataset']}, {subid}")
        pipel_dir_cur = os.path.join(pipel_dir, info_cur['dataset'], subid)
        template_img_path = os.path.join(tpl_root, "00Month", "BCP-00M-T1.nii.gz")
        t1_img_path = os.path.join(pipel_dir_cur, f"T1_BrainExtractionBrain.nii.gz")  
        t2_img_path = os.path.join(pipel_dir_cur, f"T2_BrainExtractionBrain.nii.gz")   
        if info_cur['dataset'] == 'BCP':
            t1_img_path = os.path.join(pipel_dir_cur, f"T1-skullstripped.nii.gz")  
            t2_img_path = os.path.join(pipel_dir_cur, f"T2-skullstripped.nii.gz")
            if os.path.exists(t1_img_path):
                print(f"[iBEAT] skull stripping for {info_cur['dataset']} {subid}, {t1_img_path}")
            else:
                print(f"[ANTS] for kull stripping {info_cur['dataset']} {subid}, {t1_img_path}")
                t1_img_path = os.path.join(pipel_dir_cur, f"T1_BrainExtractionBrain.nii.gz")  
                t2_img_path = os.path.join(pipel_dir_cur, f"T2_BrainExtractionBrain.nii.gz") 
            

        t1_output_path = os.path.join(pipel_dir_cur, f"T1_Brain_pad.nii.gz")
        t2_output_path = os.path.join(pipel_dir_cur, f"T2_Brain_pad.nii.gz")
        com_initialize_to_template_space(
                t1_img_path = t1_img_path,
                t2_img_path = t2_img_path,
                fix_img_path = template_img_path,
                t1_output_path = t1_output_path,
                t2_output_path= t2_output_path,
            )


## 1.3 Rigid align T2 to T1 and N4

In [9]:
import os
import sys
from tqdm import tqdm
import pandas as pd
Info_dir = '/project/4290000.01/yapwan/Projects/NeonatalNormalization/Info'
pipel_dir = "/project/4290000.01/yapwan/Projects/NeonatalNormalization/Data"
tpl_root = "/project/4290000.01/yapwan/toolbox/BCP-atlas-for_release-Ver2.0.0"
tpl_dirs = [d for d in os.listdir(tpl_root) if os.path.isdir(os.path.join(tpl_root, d))]
tpl_months = sorted([d.replace("Month", "") for d in tpl_dirs if d.endswith("Month")], key=int)

base_dir = '/project/4290000.01/yapwan/Projects/NeonatalNormalization/Script'
sys.path.append(base_dir)
from tpl_xfm_build import t1_t2_rigid_and_N4

info_age = pd.read_csv(os.path.join(Info_dir, f'BCP_HCPD_participants_0-18Y.tsv'), sep='\t')
age_bins = ['12-15M', '15-18M', '18-21M', '21-24M'] # '24-36M', '36-48M', '48-60M', '60-216M'
for age_bin in age_bins:
    print(f"Processing age bin: {age_bin}")
    subs = (info_age.loc[info_age[age_bin], ["participant_id"]].drop_duplicates().values.flatten())
    print(f"Subjects in age bin {age_bin}: {len(subs)}")
    # for subid in tqdm(subs_pos):
    for subid in tqdm(subs):
        info_cur = info_age[info_age['participant_id']==subid].iloc[0]
        print(f"{info_cur['dataset']}, {subid}")
        output_dir = os.path.join(pipel_dir, info_cur['dataset'], subid)
        input_files = {
            "T1": f"{output_dir}/T1_Brain_pad.nii.gz",
            "T2": f"{output_dir}/T2_Brain_pad.nii.gz"
        }
        # check whether input_files exist
        if not os.path.exists(input_files["T1"]):
            print(f"[WARNING] T1 file not found for {subid}, skipping...")
            continue
        t1_t2_rigid_and_N4(
            input_files=input_files,
            output_dir=output_dir,
            slurm=True
        )


Processing age bin: 12-15M
Subjects in age bin 12-15M: 30


 13%|█▎        | 4/30 [00:00<00:00, 36.86it/s]

BCP, sub-307
[INFO] Submitting job: rigN4_307
[INFO] Job script: /project/4290000.01/yapwan/Projects/NeonatalNormalization/Data/BCP/sub-307/log/rigN4_307.sh
[INFO] Command: sbatch /project/4290000.01/yapwan/Projects/NeonatalNormalization/Data/BCP/sub-307/log/rigN4_307.sh
[INFO] sbatch output: Submitted batch job 52134054
[INFO] Job ID: 52134054
BCP, sub-310
[INFO] Submitting job: rigN4_310
[INFO] Job script: /project/4290000.01/yapwan/Projects/NeonatalNormalization/Data/BCP/sub-310/log/rigN4_310.sh
[INFO] Command: sbatch /project/4290000.01/yapwan/Projects/NeonatalNormalization/Data/BCP/sub-310/log/rigN4_310.sh
[INFO] sbatch output: Submitted batch job 52134055
[INFO] Job ID: 52134055
BCP, sub-311
[INFO] Submitting job: rigN4_311
[INFO] Job script: /project/4290000.01/yapwan/Projects/NeonatalNormalization/Data/BCP/sub-311/log/rigN4_311.sh
[INFO] Command: sbatch /project/4290000.01/yapwan/Projects/NeonatalNormalization/Data/BCP/sub-311/log/rigN4_311.sh
[INFO] sbatch output: Submitted ba

 30%|███       | 9/30 [00:00<00:00, 40.58it/s]

[INFO] sbatch output: Submitted batch job 52134061
[INFO] Job ID: 52134061
BCP, sub-349
[INFO] Submitting job: rigN4_349
[INFO] Job script: /project/4290000.01/yapwan/Projects/NeonatalNormalization/Data/BCP/sub-349/log/rigN4_349.sh
[INFO] Command: sbatch /project/4290000.01/yapwan/Projects/NeonatalNormalization/Data/BCP/sub-349/log/rigN4_349.sh
[INFO] sbatch output: Submitted batch job 52134062
[INFO] Job ID: 52134062
BCP, sub-350
[INFO] Submitting job: rigN4_350
[INFO] Job script: /project/4290000.01/yapwan/Projects/NeonatalNormalization/Data/BCP/sub-350/log/rigN4_350.sh
[INFO] Command: sbatch /project/4290000.01/yapwan/Projects/NeonatalNormalization/Data/BCP/sub-350/log/rigN4_350.sh
[INFO] sbatch output: Submitted batch job 52134063
[INFO] Job ID: 52134063
BCP, sub-351
[INFO] Submitting job: rigN4_351
[INFO] Job script: /project/4290000.01/yapwan/Projects/NeonatalNormalization/Data/BCP/sub-351/log/rigN4_351.sh
[INFO] Command: sbatch /project/4290000.01/yapwan/Projects/NeonatalNormali

...

[INFO] sbatch output: Submitted batch job 52134162
[INFO] Job ID: 52134162
BCP, sub-556
[INFO] Submitting job: rigN4_556
[INFO] Job script: /project/4290000.01/yapwan/Projects/NeonatalNormalization/Data/BCP/sub-556/log/rigN4_556.sh
[INFO] Command: sbatch /project/4290000.01/yapwan/Projects/NeonatalNormalization/Data/BCP/sub-556/log/rigN4_556.sh
[INFO] sbatch output: Submitted batch job 52134163
[INFO] Job ID: 52134163
BCP, sub-558
[INFO] Submitting job: rigN4_558
[INFO] Job script: /project/4290000.01/yapwan/Projects/NeonatalNormalization/Data/BCP/sub-558/log/rigN4_558.sh
[INFO] Command: sbatch /project/4290000.01/yapwan/Projects/NeonatalNormalization/Data/BCP/sub-558/log/rigN4_558.sh
[INFO] sbatch output: Submitted batch job 52134164
[INFO] Job ID: 52134164
BCP, sub-559
[INFO] Submitting job: rigN4_559
[INFO] Job script: /project/4290000.01/yapwan/Projects/NeonatalNormalization/Data/BCP/sub-559/log/rigN4_559.sh
[INFO] Command: sbatch /project/4290000.01/yapwan/Projects/NeonatalNormali

 90%|█████████ | 27/30 [00:00<00:00, 57.63it/s]

[INFO] sbatch output: Submitted batch job 52134169
[INFO] Job ID: 52134169
BCP, sub-576
[INFO] Submitting job: rigN4_576
[INFO] Job script: /project/4290000.01/yapwan/Projects/NeonatalNormalization/Data/BCP/sub-576/log/rigN4_576.sh
[INFO] Command: sbatch /project/4290000.01/yapwan/Projects/NeonatalNormalization/Data/BCP/sub-576/log/rigN4_576.sh
[INFO] sbatch output: Submitted batch job 52134170
[INFO] Job ID: 52134170
BCP, sub-581


100%|██████████| 30/30 [00:00<00:00, 55.72it/s]

[INFO] Submitting job: rigN4_581
[INFO] Job script: /project/4290000.01/yapwan/Projects/NeonatalNormalization/Data/BCP/sub-581/log/rigN4_581.sh
[INFO] Command: sbatch /project/4290000.01/yapwan/Projects/NeonatalNormalization/Data/BCP/sub-581/log/rigN4_581.sh
[INFO] sbatch output: Submitted batch job 52134171
[INFO] Job ID: 52134171
BCP, sub-585
[INFO] Submitting job: rigN4_585
[INFO] Job script: /project/4290000.01/yapwan/Projects/NeonatalNormalization/Data/BCP/sub-585/log/rigN4_585.sh
[INFO] Command: sbatch /project/4290000.01/yapwan/Projects/NeonatalNormalization/Data/BCP/sub-585/log/rigN4_585.sh
[INFO] sbatch output: Submitted batch job 52134172
[INFO] Job ID: 52134172
BCP, sub-605
[INFO] Submitting job: rigN4_605
[INFO] Job script: /project/4290000.01/yapwan/Projects/NeonatalNormalization/Data/BCP/sub-605/log/rigN4_605.sh
[INFO] Command: sbatch /project/4290000.01/yapwan/Projects/NeonatalNormalization/Data/BCP/sub-605/log/rigN4_605.sh
[INFO] sbatch output: Submitted batch job 52134

# 2. Direct warp to age-specific and adult tpl

For each subj, 3 tpls, 2 xfms, 6 warped totally

In [ ]:
# Have alread run the direct registration to 216M for 17 subjects. 
# Load 17 subjects
import pandas as pd
import os
import sys
base_dir  = "/project/4290000.01/yapwan/Projects/NeonatalNormalization/Script"
Info_dir  = "/project/4290000.01/yapwan/Projects/NeonatalNormalization/Info"
pipel_dir = "/project/4290000.01/yapwan/Projects/NeonatalNormalization/Data"
tpl_root  = "/project/4290000.01/yapwan/toolbox/BCP-atlas-for_release-Ver2.0.0"
sys.path.append(base_dir)
from tpl_xfm_build import multimodal_register_pipeline
dataset_subs = pd.read_csv(os.path.join(Info_dir, "subjects_08M.tsv"), sep="\t")
print(f"Subjects: {len(dataset_subs)}")

# ---------- step 1: direct registration to 216M ----------
xfm_mods = ['T1T2','T1']

for _, row in dataset_subs.iterrows():
    dataset, subid = row["dataset"], row["participant_id"]
    output_dir  = os.path.join(pipel_dir, dataset, subid)
    input_files = {
        "T1": os.path.join(output_dir, "T1_Brain_pad_N4.nii.gz"),
        "T2": os.path.join(output_dir, "T2_Brain_pad_rigid2T1_N4.nii.gz"),
    }
    for xfm in xfm_mods:
        for TPL_FIX in ["08", "216"]:
            print(f"\n=== Step 1: direct registration to {TPL_FIX}M ===")
            multimodal_register_pipeline(
                modalities   = xfm,
                input_files  = input_files,
                tpl_root     = tpl_root,
                tpl_month    = TPL_FIX,
                output_dir   = output_dir,
                steps        = [1, 2, 3, 4],
                slurm        = True
            )


## 2.1 Visual QC and R to tpl

In [ ]:
# QC check of registration results
import os
import pandas as pd
import ants
import matplotlib.pyplot as plt
from tqdm import tqdm
import numpy as np

pipel_dir = "/project/4290000.01/yapwan/Projects/NeonatalNormalization/Data"
Info_dir = '/project/4290000.01/yapwan/Projects/NeonatalNormalization/Info'
result_dir = "/project/4290000.01/yapwan/Projects/NeonatalNormalization/Result/QC"
os.makedirs(result_dir, exist_ok=True)
tpl_root = "/project/4290000.01/yapwan/toolbox/BCP-atlas-for_release-Ver2.0.0"

modalities = 'T1T2'
age_bins = [ 
             '0-1M', '1-2M', '2-3M', '3-4M', '4-5M', '5-6M', '6-7M', '7-8M', '8-9M', '9-10M',
             '10-11M', '11-12M', 
             '12-15M', '15-18M', '18-21M', '21-24M',
             '24-36M', '36-48M', 
             '48-60M','60-216M'
             ]
info_age = pd.read_csv(os.path.join(Info_dir, f'BCP_HCPD_participants_0-18Y.tsv'), sep='\t')
sub_check = []
for age_bin in age_bins:
    print(f"Processing age bin: {age_bin}")
    tpl_months = [age_bin.replace("M","").split("-")[0].zfill(2),age_bin.replace("M","").split("-")[1].zfill(2)]
    print(f'Tpl month: {tpl_months}')
    # check whether f'{age_bin}_all' column exists in info_age
    if f'{age_bin}_all' in info_age.columns:
        print(f"Column '{age_bin}_all' found in info_age, doing all subject in {age_bin}")
        age_bin = f'{age_bin}_all'
    else:
        print(f"Column '{age_bin}_all' not found in info_age, doing picked subjects only in {age_bin}")
    subs = (info_age.loc[info_age[age_bin], ["participant_id"]].drop_duplicates().values.flatten())
    print(f"Subjects in age bin {age_bin}: {len(subs)}")
    tpl_dict = {}
    msk_dict = {}
    for tpl_month in tpl_months:
        print(f"Running registration → tpl_month={tpl_month}")
        tpl_file = os.path.join(
            tpl_root, f"{tpl_month}Month", f"BCP-{tpl_month}M-T1.nii.gz"
        )
        mask_file = os.path.join(
                tpl_root, f"{tpl_month}Month", f"BCP-{tpl_month}M-Mask.nii.gz"
            )
        tpl_dict[tpl_month] = ants.image_read(tpl_file).numpy()
        msk_dict[tpl_month] = ants.image_read(mask_file).numpy()

        n_subj = len(subs)
        n_cols = len(tpl_months)
        fig, axes = plt.subplots(
            n_subj, n_cols,
            figsize=(4 * n_cols, 2.2 * n_subj),
            squeeze=False
        )

    for i, subid in tqdm(enumerate(subs)):
        info_cur = info_age[info_age['participant_id']==subid].iloc[0]
        output_dir = os.path.join(pipel_dir, info_cur['dataset'], subid)

        for j, tpl_month in enumerate(tpl_months):

            registered_img_path = os.path.join(
                output_dir,
                f"T1_resliced_to_{tpl_month}Mtpl_by_direct_{modalities}_xfm.nii.gz"
            )

            if not os.path.exists(registered_img_path):
                continue

            img_np = ants.image_read(registered_img_path).numpy()
            tpl_np = tpl_dict[tpl_month]

            ax = axes[i, j]
            ax.axis('off')

            # ---------- use consistent axial slice ----------
            # sl = img_np.shape[1] // 2
            sl = np.argmax(img_np.sum(axis=(0,2)))

            ax.imshow(
                tpl_np[:, sl, :].T,
                cmap='gray',
                origin='lower'
            )

            ax.imshow(
                img_np[:, sl, :].T,
                cmap='viridis',     
                alpha=0.4,
                origin='lower'
            )
            # Calculate R and show in left upper corner
            score = np.corrcoef(
                tpl_np[msk_dict[tpl_month]>0],
                img_np[msk_dict[tpl_month]>0]
            )[0, 1]
            ax.text(
                    0.22, 0.80,
                    f'{score:.2f}',#{eval_mode}=
                    transform=ax.transAxes,
                    ha='right',
                    va='bottom',
                    fontsize=15,
                    color='white',
                    bbox=dict(
                        facecolor='black',
                        alpha=0.5,
                        edgecolor='none',
                        pad=1.5
                    )
                )
            if score < 0.5:
                sub_check.append([info_cur['dataset'],subid])

            # ---------- column titles ----------
            if i == 0:
                ax.set_title(f'{tpl_month}M', fontsize=10)

        # ---------- row labels ----------
        axes[i, 0].text(
            -0.1, 0.5,
            f'{info_cur['dataset']}, {subid} \n{info_cur['age_years'].round(2)}',
            fontsize=8,
            va='center',
            ha='right',
            transform=axes[i, 0].transAxes
        )

    plt.tight_layout()
    plt.savefig(f"{result_dir}/{info_cur['dataset']}{age_bin}_QC.png", dpi=200)
    print(f"Saved QC figure for age bin {age_bin} at {result_dir}/{info_cur['dataset']}{age_bin}_QC.png")
    plt.close()
# write sub_check to csv
check_df = pd.DataFrame(sub_check)
check_df.columns = ['dataset', 'participant_id']
# delete repeat dataset and participant_id
check_df = check_df.drop_duplicates(subset=['dataset', 'participant_id'])
check_df.to_csv(os.path.join(result_dir, f"BCP_HCPD_0-18Y_registration_check.csv"), index=False)

# 3. Cereb mask dice score?